## Readme
input: gpx track files
requirement: way points(include start point and end point), all way points name filled, no need time

In [ ]:
pip install gpxpy mgrs geopy

In [85]:
import gpxpy
from pathlib import Path
import pandas as pd
from geopy.distance import geodesic
import mgrs
import math

path = Path(r"path_to\track.gpx")
speed = 4000 # speed on flat ground in m/h

In [87]:
def calculate_bearing(lat1, lon1, lat2, lon2):
    """Returns bearing in degrees from point A to point B."""
    lat1 = math.radians(lat1)
    lat2 = math.radians(lat2)
    diff_lon = math.radians(lon2 - lon1)

    x = math.sin(diff_lon) * math.cos(lat2)
    y = math.cos(lat1) * math.sin(lat2) - \
        math.sin(lat1) * math.cos(lat2) * math.cos(diff_lon)

    initial_bearing = math.atan2(x, y)
    bearing_deg = (math.degrees(initial_bearing) + 360) % 360

    return round(bearing_deg, 2)

In [37]:
with open(path, 'r', encoding='utf-8') as gpxf:
    gpx = gpxpy.parse(gpxf)

pts = []
for track in gpx.tracks:
    for segment in track.segments:
        for point in segment.points:
            pts.append([point.latitude,
                       point.longitude,
                       int(point.elevation),])

df = pd.DataFrame(pts, columns=['lat', 'lon', 'ele'])

In [39]:
m = mgrs.MGRS()
df["MGRS"] = df.apply(lambda row: m.toMGRS(row["lat"], row["lon"], MGRSPrecision=3), axis=1)

In [ ]:
df.tail()

In [43]:
waypoints = [{'name': wp.name, 'lat': wp.latitude, 'lon': wp.longitude} for wp in gpx.waypoints]

In [98]:
from scipy.spatial import cKDTree
import numpy as np
# KDTree for fast nearest neighbor search
track_coords = df[['lat', 'lon']].to_numpy()
tree = cKDTree(track_coords)
used_indices = set()
results = []

for wp in waypoints:
    wp_coord = np.array([[wp['lat'], wp['lon']]])
    
    # Get nearest n to try skipping duplicates
    dists, idxs = tree.query(wp_coord, k=5)
    
    for dist, idx in zip(dists[0], idxs[0]):
        if idx not in used_indices:
            used_indices.add(idx)
            track_point = df.iloc[idx]
            geo_dist = geodesic((wp['lat'], wp['lon']), (track_point.lat, track_point.lon)).meters

            results.append({
                'waypoint_name': wp['name'],
                'waypoint_lat': wp['lat'],
                'waypoint_lon': wp['lon'],
                'nearest_lat': track_point.lat,
                'nearest_lon': track_point.lon,
                'MGRS': track_point.MGRS,
                'nearest_ele': track_point.ele,
                'dist_m': geo_dist,
                'track_idx': idx,
            })
            break

# Create DataFrame
matched_df = pd.DataFrame(results)
# Sort by track index to follow actual track progression
matched_df = matched_df.sort_values('track_idx').reset_index(drop=True)

matched_df['ele_diff'] = matched_df['nearest_ele'].diff()
matched_df['ele_up'] = matched_df['ele_diff'].apply(lambda x: int(x) if x > 0 else 0)
matched_df['ele_down'] = matched_df['ele_diff'].apply(lambda x: -int(x) if x < 0 else 0)

# Distance between waypoints
matched_df['dist'] = matched_df.apply(
    lambda row: geodesic((row.nearest_lat, row.nearest_lon), 
                         (matched_df.iloc[row.name - 1].nearest_lat, 
                          matched_df.iloc[row.name - 1].nearest_lon)).meters 
    if row.name > 0 else 0, axis=1
)
# speed for time consumption
speed_m_per_min = speed / 60

matched_df['flat_t'] = matched_df['dist'] / speed_m_per_min
matched_df['extra_time_min'] = (matched_df['ele_up'] / 20) * 3
matched_df['total_t'] = matched_df['flat_t'] + matched_df['extra_time_min']

# bearing calculation
matched_df['bearing'] = matched_df.apply(
    lambda row: calculate_bearing(
        matched_df.iloc[row.name - 1]['nearest_lat'], 
        matched_df.iloc[row.name - 1]['nearest_lon'], 
        row['nearest_lat'], 
        row['nearest_lon']) if row.name > 0 else None, axis=1)

In [ ]:
matched_df[['waypoint_name', 'waypoint_lat', 'waypoint_lon', 'nearest_lat', 'nearest_lon']].tail()

In [102]:
columns_order = ['waypoint_name', 'nearest_lat', 'nearest_lon', 'MGRS', 'bearing', 'dist', 'ele_up', 'ele_down', 'total_t']
output_df = matched_df[columns_order]

# Export to CSV
output_df.to_csv("hike_plan.csv", index=False)